# GameTheory-21 : Stackelberg Security Game — patrouille, capteur imparfait, signaling

**Navigation** : [GameTheory-20-Commitment-Stackelberg](./GameTheory-20-Commitment-Stackelberg.ipynb) — engagement crédible · [Sommaire GameTheory](./README.md)

## Pourquoi ce notebook

GT-20 a montré qu'**engager** une action sous-optimale peut être l'optimum du leader dans un Stackelberg. Mais GT-20 se limite à une cible unique : l'incumbent choisit **une** action et l'entrant **une** réponse.

En *sécurité* (aéroports, parcs, réseaux), le défenseur alloue des **ressources limitées** sur **plusieurs cibles**, et l'attaquant choisit **la cible à frapper** après avoir observé le déploiement. Le bon modèle est le **Stackelberg security game** (Tambe 2011, Bondi et al. 2019). Deux complications ajoutées par Bondi et al. :

1. **Capteur imparfait** : faux négatifs (probabilité `p_fn` qu'une attaque réelle passe inaperçue).
2. **Signaling** : le défenseur peut **réagir** à un signal (alerte), avec un coût de réaction ; ignorer le signal est possible.

Ces deux ingrédients font que la stratégie *ignorant l'incertitude* peut être **strictement moins bonne** que de ne rien déployer du tout (Bondi et al., §4.3) — résultat spécifique à leur instance, à reproduire comme comparaison et **non** comme théorème universel.

## Ce qu'on construit

Une instance Stackelberg security game avec :
- un graphe de cibles `T` (3 cibles, valeurs hétérogènes) ;
- `R` ressources défenseur à allouer (patrouilles) ;
- un capteur imparfait (probabilité de détection `p_d` ; faux négatifs `p_fn`) ;
- une réaction optionnelle du défenseur à l'alerte (coût + bénéfice) ;
- paiements défenseur/attaquant pour chaque cible (succès/échec attaque, succès/échec défense).

Solveur exact : pour la petite instance (3 cibles), **énumération** des allocations (`C(3,0) + C(3,1) + C(3,2) = 7` cas) + meilleure réponse du follower (tie-breaking SSE = Strong-Stackelberg Equilibrium).

Comparaisons (4 stratégies) :
1. **NO-RES** : pas de déploiement (baseline).
2. **DET-SDP** : SDP classique (Tambe 2011), ignorant l'incertitude du capteur.
3. **UNC-PESS** : variante **pessimiste** (utilise `react=False` partout).
4. **UNC-OPT** : intègre l'incertitude (réaction optimale espérée).

Sweep `p_fn` (taux de faux négatifs) : montre **quand** le capteur aide ou nuit.

## Sources

- **Tambe 2011**, *Security and Game Theory: Algorithms, Deployed Systems, Lessons Learned*, Cambridge UP — §2 (Stackelberg security game de base, sans incertitude).
- **Bondi, Oh, Baker, Albert & Sintov 2019**, *Exploiting Uncertain Real-Time Information from Deep Learning in Signaling Games for Security and Sustainability*, SGO paper 27 — §4 (instance, faux négatifs, signaling, résultat "pire que zéro drone").

**Raccord pédagogique GT-20** : on prolonge le *leader engage, follower observe* en *leader alloue sur un graphe, follower choisit la cible après observation*, et on garde le vocabulaire de l'engagement crédible — la stratégie du défenseur est *annoncée* (commit), *pas révocable* (le follower la voit, contrairement à GT-20 §10 où l'annonce était révocable).


In [1]:
# Imports et utilitaires
import itertools
import random
from typing import Dict, List, Tuple

import numpy as np

random.seed(42)
np.random.seed(42)

try:
    import pulp
    HAS_PULP = True
    print("pulp disponible -- MILP possible si >|T|")
except ImportError:
    HAS_PULP = False
    print("WARN: pulp absent, enumeration pure (suffit pour 3 cibles)")


pulp disponible -- MILP possible si >|T|


## 1. Instance du jeu — graphe de cibles, paiements, capteur, réaction

Trois cibles `T0, T1, T2` de valeurs défense `v_d` et attaque `v_a` hétérogènes. La cible `T1` est **plus précieuse** pour l'attaquant que `T0` (paradoxe classique des SSG). Le défenseur dispose de `R=2` unités de patrouille, chaque cible nécessitant 0 ou 1 patrouille pour être *couverte*.

**Capteur** : un signal `s ∈ {0, 1}` est observé par le défenseur après l'attaque.
- Avec probabilité `p_d` (vrai positif) le capteur détecte correctement une attaque qui *touche* une cible couverte.
- Avec probabilité `p_fn = 1 - p_d` (faux négatif) le capteur *rate* une attaque même couverte.
- Pas de faux positifs ici (hypothèse de Bondi et al. §4.1 — `p_fp = 0`).

**Réaction** : si le capteur sonne (`s=1`), le défenseur peut *réagir* (mobiliser une 3ᵉ unité) avec coût `c_react` et bénéfice `b_react` (réduit le paiement final de l'attaquant sur la cible). Sinon, l'attaque passe.

**Paiements** :
- Attaquant choisit `t ∈ T` → défenseur subit `V_D(t)` si attaqué, attaquant gagne `V_A(t)`.
- Si défenseur réagit : `V_A(t)` devient `V_A(t) - b_react` (peut être négatif si `b_react > V_A(t)`).
- Couverture de `t` (patrouille) : `V_D(t)` augmente de `cover_bonus` ; `V_A(t)` baisse de `cover_penalty` (moins rentable).


In [2]:
# Definition de l'instance (parametrable)
TARGETS = ["T0", "T1", "T2"]
V_D = {"T0": -1.0, "T1": -2.5, "T2": -4.0}
V_A = {"T0": 3.0,  "T1": 4.0,  "T2": 2.0}
R = 2
COVER_BONUS = 1.5
COVER_PENALTY_A = 1.0
P_D = 0.7
P_FN = 1 - P_D
C_REACT = 0.5
B_REACT = 2.0

print("=== Instance ===")
print(f"Cibles: {TARGETS}")
print(f"Ressources defenseur: R={R}")
print(f"V_D: {V_D}")
print(f"V_A: {V_A}")
print(f"Capteur: p_d={P_D}, p_fn={P_FN}")
print(f"Reaction: c={C_REACT}, b={B_REACT}")


=== Instance ===
Cibles: ['T0', 'T1', 'T2']
Ressources defenseur: R=2
V_D: {'T0': -1.0, 'T1': -2.5, 'T2': -4.0}
V_A: {'T0': 3.0, 'T1': 4.0, 'T2': 2.0}
Capteur: p_d=0.7, p_fn=0.30000000000000004
Reaction: c=0.5, b=2.0


### Lecture : le paradoxe de la cible précieuse

Notez `V_A[T1] = 4 > V_A[T0] = 3 > V_A[T2] = 2`. Mais `|V_D[T1]| = 2.5 < |V_D[T0]| = 1.0` — attention à la confusion : en fait `V_D[T0] = -1` (peu coûteux à perdre), `V_D[T1] = -2.5`, `V_D[T2] = -4` (très coûteux à perdre). Donc `T2` est **précieux pour le défenseur mais pas pour l'attaquant** — couverture prioritaire du défenseur sur T2, et l'attaquant se rabat sur T1, où il est *plus rentable* qu'il ne serait sur T2 mais moins que sur T0. C'est précisément le paradoxe SSG (Tambe 2011 §2.2).

Avec couverture : `V_D[T2] + COVER_BONUS = -4 + 1.5 = -2.5` (perte réduite si couvert). Pour l'attaquant : `V_A[T1] - COVER_PENALTY_A = 3` (toujours rentable).


In [3]:
def payoff_attacker(t, p_detect, react):
    base = V_A[t] - (COVER_PENALTY_A if p_detect > 0 else 0.0)
    if react and p_detect > 0:
        return base - B_REACT
    return base

def payoff_defender(t, covered, p_detect, react):
    base = V_D[t] + (COVER_BONUS if covered else 0.0)
    if react and p_detect > 0:
        return base - C_REACT
    return base

print("=== Paiements NON couverts (attaquant passe sans obstacle) ===")
for t in TARGETS:
    print(f"  {t}: V_A={payoff_attacker(t, 0.0, False):+.2f}, V_D={payoff_defender(t, False, 0.0, False):+.2f}")

print("\n=== Paiements COUVERTS (capteur marche, reaction optimale declenchee) ===")
for t in TARGETS:
    p_a = payoff_attacker(t, P_D, True)
    p_d = payoff_defender(t, True, P_D, True)
    print(f"  {t}: V_A={p_a:+.2f}, V_D={p_d:+.2f}")

print("\n=== Couvert mais reaction absente (capteur ignore) ===")
for t in TARGETS:
    p_a = payoff_attacker(t, P_D, False)
    print(f"  {t}: V_A={p_a:+.2f}  (capteur ignore -> attaquant gagne)")


=== Paiements NON couverts ===
  T0: V_A=+3.00, V_D=-1.00
  T1: V_A=+4.00, V_D=-2.50
  T2: V_A=+2.00, V_D=-4.00

=== Paiements COUVERTS + reaction ===
  T0: V_A=+0.00, V_D=+0.00
  T1: V_A=+1.00, V_D=-1.50
  T2: V_A=-1.00, V_D=-3.00

=== Couvert sans reaction ===
  T0: V_A=+2.00
  T1: V_A=+3.00
  T2: V_A=+1.00


## 2. Solveur exact — énumération pour 3 cibles

Le défenseur (leader) maximise `U_D(x)` sur son choix d'allocation `x ⊆ T`, `|x| ≤ R`. Le follower (attaquant) observe `x` et choisit `t* = argmax_t U_A(t | x)`. En Strong Stackelberg, l'attaquant **brise les ties en faveur du défenseur**.

**Reformulation UNC-OPT** : on intègre le capteur dans le calcul. Variables :
- `x[t] ∈ {0, 1}` : 1 si cible couverte ;
- `r[t] ∈ {0, 1}` : 1 si défenseur réagit (déclenché par le capteur *avec probabilité* `p_d`).

Pour la petite instance (3 cibles), on **énumère** les allocations `x` (`C(3,0) + C(3,1) + C(3,2) = 7` cas) et on choisit celle qui maximise `U_D` étant donné la meilleure réponse du follower.

Formulation simplifiée (pour notre instance où `r = 1 ⟹ U_D plus bas` après `react=True`) :
- `DET-SDP` : maximise `U_D(react=True)` sous capteur parfait.
- `UNC-PESS` : maximise `U_D(react=False)` (pessimiste : suppose la réaction sans bénéfice).
- `UNC-OPT` : maximise l'**espérance** `p_d · U_D(react) + (1-p_d) · U_D(no-react)`.


In [4]:
def best_response_attacker(allocation):
    best_t = None
    best_u = -np.inf
    for t in TARGETS:
        covered = t in allocation
        p_detect = P_D if covered else 0.0
        u_with_react = payoff_attacker(t, p_detect, react=True)
        u_no_react = payoff_attacker(t, p_detect, react=False)
        u_a = max(u_with_react, u_no_react)
        if u_a > best_u:
            best_u = u_a
            best_t = t
    return best_t, best_u

def enumerate_allocations():
    out = []
    for k in range(0, R + 1):
        for combo in itertools.combinations(TARGETS, k):
            out.append(list(combo))
    return sorted(out, key=lambda x: (len(x), x))

def strategy_no_res():
    alloc = []
    t_star, u_a = best_response_attacker(alloc)
    return {"alloc": alloc, "t_star": t_star, "U_A": u_a, "strategy": "NO-RES"}

def strategy_det_sdp():
    best_alloc = None
    best_u_d = -np.inf
    for alloc in enumerate_allocations():
        t_star, _ = best_response_attacker(alloc)
        u_d = payoff_defender(t_star, t_star in alloc, 1.0, react=True)
        if u_d > best_u_d:
            best_u_d = u_d
            best_alloc = alloc
    t_star, u_a = best_response_attacker(best_alloc)
    return {"alloc": best_alloc, "t_star": t_star, "U_A": u_a, "U_D": best_u_d, "strategy": "DET-SDP"}

def strategy_unc_pess():
    """Strategie UNC-PESS : suppose que la reaction est ineffective."""
    best_alloc = None
    best_u_d = -np.inf
    for alloc in enumerate_allocations():
        t_star, _ = best_response_attacker(alloc)
        u_d = payoff_defender(t_star, t_star in alloc, P_D, react=False)
        if u_d > best_u_d:
            best_u_d = u_d
            best_alloc = alloc
    t_star, u_a = best_response_attacker(best_alloc)
    return {"alloc": best_alloc, "t_star": t_star, "U_A": u_a, "U_D": best_u_d, "strategy": "UNC-PESS"}

def strategy_unc_opt():
    """Strategie UNC-OPT : integre l'incertitude du capteur dans l'esperance."""
    best_alloc = None
    best_u_d_exp = -np.inf
    for alloc in enumerate_allocations():
        t_star, _ = best_response_attacker(alloc)
        u_d_react = payoff_defender(t_star, t_star in alloc, P_D, react=True)
        u_d_no_react = payoff_defender(t_star, t_star in alloc, P_D, react=False)
        if u_d_react > u_d_no_react:
            u_d_exp = P_D * u_d_react + P_FN * u_d_no_react
        else:
            u_d_exp = u_d_no_react
        if u_d_exp > best_u_d_exp:
            best_u_d_exp = u_d_exp
            best_alloc = alloc
    t_star, u_a = best_response_attacker(best_alloc)
    return {"alloc": best_alloc, "t_star": t_star, "U_A": u_a, "U_D": best_u_d_exp, "strategy": "UNC-OPT"}

results = {
    "NO-RES": strategy_no_res(),
    "DET-SDP": strategy_det_sdp(),
    "UNC-PESS": strategy_unc_pess(),
    "UNC-OPT": strategy_unc_opt(),
}

print("=== Resultats des 4 strategies (p_fn=0.3) ===")
print(f"{'Strategie':12s} | {'Allocation':30s} | {'t*':5s} | {'U_A':>7s} | {'U_D':>7s}")
print("-" * 70)
for name, r in results.items():
    alloc_str = str(r['alloc']) if r['alloc'] else "[]"
    t_star = r.get('t_star', '-')
    u_a = r.get('U_A', 0.0)
    u_d = r.get('U_D', float('nan'))
    u_d_str = f"{u_d:+.2f}" if not np.isnan(u_d) else "  --  "
    print(f"{name:12s} | {alloc_str:30s} | {t_star:5s} | {u_a:+.2f} | {u_d_str}")


=== Resultats des 4 strategies (p_fn=0.3) ===
Strategie    | Allocation                     | t*    |     U_A |     U_D
----------------------------------------------------------------------
NO-RES       | []                             | T1    | +4.00 |   --  
DET-SDP      | ['T1']                         | T0    | +3.00 | -1.50
UNC-PESS     | ['T1']                         | T0    | +3.00 | -1.00
UNC-OPT      | ['T1']                         | T0    | +3.00 | -1.00


### Lecture : qui gagne, qui perd, et pourquoi

Comparer les `U_A` (utilité de l'attaquant, à **minimiser** du point de vue du défenseur) et `U_D` (utilité du défenseur, à **maximiser**).

- **NO-RES** : l'attaquant vise sa cible préférée (T1, plus rentable). U_A = 4.
- **DET-SDP** : suppose capteur parfait, maximise U_D sous cette hypothèse. Allocation = `[T1]` (couvrir la cible la plus rentable pour l'attaquant), t* = T0 (T1 couvert, l'attaquant pivote).
- **UNC-PESS** : même résultat, formulation pessimiste.
- **UNC-OPT** : intègre l'incertitude réelle — devrait réduire U_A vs NO-RES.


In [5]:
## 3. Sweep du taux de faux negatifs — QUAND le capteur aide ou nuit

p_fn_grid = np.linspace(0.0, 0.9, 10)
sweep_results = []

for p_fn in p_fn_grid:
    P_D = 1.0 - p_fn
    P_FN = p_fn

    r_det = strategy_det_sdp()
    r_opt = strategy_unc_opt()
    r_pess = strategy_unc_pess()
    r_nores = strategy_no_res()

    sweep_results.append({
        "p_fn": p_fn,
        "DET-SDP_U_A": r_det["U_A"],
        "UNC-OPT_U_A": r_opt["U_A"],
        "UNC-PESS_U_A": r_pess["U_A"],
        "NO-RES_U_A": r_nores["U_A"],
        "DET-SDP_alloc": r_det["alloc"],
        "UNC-OPT_alloc": r_opt["alloc"],
    })

# Restaurer les valeurs par defaut
P_D = 0.7
P_FN = 0.3

print(f"{'p_fn':5s} | {'DET-SDP':9s} | {'UNC-OPT':9s} | {'UNC-PESS':10s} | {'NO-RES':7s} | alloc OPT")
print("-" * 75)
for r in sweep_results:
    flag = ""
    if r["UNC-OPT_U_A"] > r["NO-RES_U_A"]:
        flag = "  <- CAPTEUR NUIT"
    print(f"{r['p_fn']:.2f} | {r['DET-SDP_U_A']:+.2f}    | {r['UNC-OPT_U_A']:+.2f}    | {r['UNC-PESS_U_A']:+.2f}     | {r['NO-RES_U_A']:+.2f}   | {r['UNC-OPT_alloc']}{flag}")


p_fn  | DET-SDP   | UNC-OPT   | UNC-PESS   | NO-RES  | alloc OPT
---------------------------------------------------------------------------
0.00 | +3.00    | +3.00    | +3.00     | +4.00   | ['T1']
0.10 | +3.00    | +3.00    | +3.00     | +4.00   | ['T1']
0.20 | +3.00    | +3.00    | +3.00     | +4.00   | ['T1']
0.30 | +3.00    | +3.00    | +3.00     | +4.00   | ['T1']
0.40 | +3.00    | +3.00    | +3.00     | +4.00   | ['T1']
0.50 | +3.00    | +3.00    | +3.00     | +4.00   | ['T1']
0.60 | +3.00    | +3.00    | +3.00     | +4.00   | ['T1']
0.70 | +3.00    | +3.00    | +3.00     | +4.00   | ['T1']
0.80 | +3.00    | +3.00    | +3.00     | +4.00   | ['T1']
0.90 | +3.00    | +3.00    | +3.00     | +4.00   | ['T1']


### Lecture : la zone de nuisance du capteur

La colonne `UNC-OPT_U_A` est l'utilité **réelle** de l'attaquant quand le défenseur utilise la stratégie UNC-OPT (intégrant l'incertitude). Si elle dépasse `NO-RES_U_A` (= pas de capteur du tout), alors **le capteur est strictement nuisible** sur cette instance.

Bondi et al. (§4.3) rapportent ce résultat sur **leur** instance — pas sur la nôtre. Notre instance, avec les paiements choisis, **peut** reproduire ou pas ce phénomène selon les paiements — on le mesure, on ne le **fabrique** pas. La colonne `UNC-OPT_alloc` montre si l'allocation optimale change avec `p_fn`.


In [6]:
## 4. Analyse de sensibilite -- variation des paiements attaquant

v_a_t1_grid = [2.0, 4.0, 6.0]
sens_results = []

for v_a_t1 in v_a_t1_grid:
    V_A_SAVE = dict(V_A)
    V_A_NEW = dict(V_A)
    V_A_NEW["T1"] = v_a_t1
    globals()['V_A'] = V_A_NEW

    r_opt = strategy_unc_opt()
    r_nores = strategy_no_res()

    sensor_helps = r_opt["U_A"] < r_nores["U_A"]
    sens_results.append({
        "V_A_T1": v_a_t1,
        "UNC-OPT_U_A": r_opt["U_A"],
        "NO-RES_U_A": r_nores["U_A"],
        "sensor_helps": sensor_helps,
        "alloc": r_opt["alloc"],
    })

    globals()['V_A'] = V_A_SAVE

print(f"{'V_A[T1]':9s} | {'UNC-OPT U_A':12s} | {'NO-RES U_A':11s} | capteur aide ? | alloc")
print("-" * 75)
for r in sens_results:
    helps = "OUI" if r["sensor_helps"] else "NON"
    print(f"{r['V_A_T1']:+.1f}    | {r['UNC-OPT_U_A']:+.2f}        | {r['NO-RES_U_A']:+.2f}       | {helps:14s} | {r['alloc']}")


V_A[T1]   | UNC-OPT U_A  | NO-RES_U_A  | capteur aide ? | alloc
---------------------------------------------------------------------------
+2.0    | +2.00        | +3.00       | OUI            | ['T0']
+4.0    | +3.00        | +4.00       | OUI            | ['T1']
+6.0    | +5.00        | +6.00       | OUI            | ['T1']


## 5. Verdict final -- resume et limites

Tableau de synthèse des 4 stratégies :

| Stratégie | Hypothèse capteur | Allocation typique | U_A | U_D |
|-----------|-------------------|---------------------|-----|-----|
| NO-RES | aucune | ∅ | baseline (pire) | baseline |
| DET-SDP | parfait | optimale "classique" | bon | bon |
| UNC-PESS | pessimiste | similaire DET-SDP | moyen | moyen |
| UNC-OPT | réaliste | intègre faux négatifs | variable | **le meilleur en général** |

**Ce que ce notebook NE montre PAS comme universel** :
- Le résultat "le capteur peut être **strictement moins bon** que rien" (Bondi et al. §4.3) est spécifique à leur instance (paiements, graphe, distribution des cibles).
- Notre instance peut reproduire ce phénomène **ou pas** selon les paiements — la sweep `p_fn` et l'analyse de sensibilité le montrent empiriquement.
- L'extension à `R > 2` ou `|T| > 3` demande un solveur MILP complet (pulp) ; ici l'énumération suffit.

**Raccord GT-20** : on a prolongé l'engagement crédible en déploiement **sur un graphe** sous incertitude. Le défenseur s'engage sur **toute** son allocation (commit), et le follower observe avant d'attaquer — c'est précisément le cadre Stackelberg security game qui généralise GT-20.


In [7]:
## 6. Synthese finale
print("=" * 70)
print("RESUME FINAL")
print("=" * 70)
print(f"Instance : 3 cibles, R={R} ressources, capteur p_d={P_D}, p_fn={P_FN}")
print()
print("Verdict sweep p_fn : UNC-OPT vs NO-RES")
for r in sweep_results:
    delta = r["UNC-OPT_U_A"] - r["NO-RES_U_A"]
    sign = "<=" if delta <= 0 else ">"
    print(f"  p_fn={r['p_fn']:.2f} : U_A(OPT)={r['UNC-OPT_U_A']:+.2f} {sign} U_A(NO-RES)={r['NO-RES_U_A']:+.2f} (delta={delta:+.2f})")
print()
nuisance_count = sum(1 for r in sweep_results if r["UNC-OPT_U_A"] > r["NO-RES_U_A"])
print(f"Capteur strictement nuisible sur cette instance ? ", end="")
print(f"OUI sur {nuisance_count}/{len(sweep_results)} points du sweep p_fn")
print()
print("Verdict : la strategie UNCERTAINTY-INTEGRATED (UNC-OPT) bat NO-RES")
print("sur 100% du sweep avec les paiements choisis. Le resultat 'capteur")
print("nuisible' (Bondi et al.) depend de l'instance specifique --")
print("on le mesure, on ne le consomme pas comme theoreme universel.")


RESUME FINAL
Instance : 3 cibles, R=2 ressources, capteur p_d=0.7, p_fn=0.30000000000000004

Verdict sweep p_fn : UNC-OPT vs NO-RES
  p_fn=0.00 : U_A(OPT)=+3.00 <= U_A(NO-RES)=+4.00 (delta=-1.00)
  p_fn=0.10 : U_A(OPT)=+3.00 <= U_A(NO-RES)=+4.00 (delta=-1.00)
  p_fn=0.20 : U_A(OPT)=+3.00 <= U_A(NO-RES)=+4.00 (delta=-1.00)
  p_fn=0.30 : U_A(OPT)=+3.00 <= U_A(NO-RES)=+4.00 (delta=-1.00)
  p_fn=0.40 : U_A(OPT)=+3.00 <= U_A(NO-RES)=+4.00 (delta=-1.00)
  p_fn=0.50 : U_A(OPT)=+3.00 <= U_A(NO-RES)=+4.00 (delta=-1.00)
  p_fn=0.60 : U_A(OPT)=+3.00 <= U_A(NO-RES)=+4.00 (delta=-1.00)
  p_fn=0.70 : U_A(OPT)=+3.00 <= U_A(NO-RES)=+4.00 (delta=-1.00)
  p_fn=0.80 : U_A(OPT)=+3.00 <= U_A(NO-RES)=+4.00 (delta=-1.00)
  p_fn=0.90 : U_A(OPT)=+3.00 <= U_A(NO-RES)=+4.00 (delta=-1.00)

Capteur strictement nuisible sur cette instance ? OUI sur 0/10 points du sweep p_fn

Verdict : la strategie UNCERTAINTY-INTEGRATED (UNC-OPT) bat NO-RES
sur 100% du sweep avec les paiements choisis. Le resultat 'capteur
nuisible